# Sampling 與不確定性 (Sampling & Uncertainty)

## 模組脈絡：隨機取樣是 LLM「不可控」的根源

本筆記是 **01-不可控的根源** 的概念核心。前一章看到 `temperature`、`max_output_tokens`、`instructions/input` 這些參數，這一章解釋它們**為何**存在：LLM 每一步都在一個機率分布上**取樣（sampling）**下一個 token，這個隨機性正是「同樣的輸入、不同的輸出」的來源。

理解這層機制，才能理解後續所有模組在做的事——**用工程手段把不可控收斂回可控**：
- `temperature` / `top_p`：直接調節隨機性（本章）
- 結構化輸出、自我一致性（self-consistency）、best-of-N：在輸出層收斂（模組 02、03、07）

我們會用 **OpenAI / Claude / Gemini 三家並排**，看相同概念在不同 API 的差異。OpenAI 範例以 2026 新專案建議的 **Responses API** 為主。

## 0. 環境設定

In [ ]:
# 套件由 prompt-engineering/pyproject.toml 定版（uv sync）
from dotenv import load_dotenv
import os
from collections import Counter
import math

load_dotenv()
for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"):
    if not os.getenv(k):
        print(f"(提醒) 尚未設定 {k}，該 provider 的範例會跳過")

from openai import OpenAI

openai_client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. 為什麼會「不確定」？

模型輸出每個 token 前，會對整個詞彙表算出一個機率分布。`temperature` 控制這個分布的「銳利度」：

- **temperature = 0**：幾乎總是挑機率最高的 token（接近貪婪解碼）→ 輸出最穩定。
- **temperature 越高**：分布越平坦，低機率 token 也可能被選中 → 輸出越多樣、越不可預測。

下面用同一個開放式問題，比較低溫與高溫各跑幾次的差異。

In [ ]:
def ask_openai(prompt, temperature, runs=1, model=OPENAI_MODEL):
    """用 Responses API 重複取樣 runs 次，回傳文字列表。"""
    outputs = []
    for _ in range(runs):
        resp = openai_client.responses.create(
            model=model,
            input=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_output_tokens=30,
        )
        outputs.append(resp.output_text.strip())
    return outputs

prompt = "用一個詞形容台北的天氣，只回那個詞。"
print("temperature=0 :", ask_openai(prompt, 0, runs=5))
print("temperature=1.2:", ask_openai(prompt, 1.2, runs=5))

## 2. 實證：多次執行的一致性

把「不可控」量化——同一個提示跑 N 次，數出現幾種不同答案。這就是 self-consistency 的觀測基礎。

In [ ]:
def consistency_report(prompt, temperature, runs=10):
    answers = ask_openai(prompt, temperature, runs=runs)
    counter = Counter(answers)
    print(f"temperature={temperature}: {len(counter)} 種相異答案 / {runs} 次")
    for ans, cnt in counter.most_common():
        print(f"  {cnt:>2}x  {ans!r}")

q = "請從 1 到 100 之間選一個數字，只回數字。"
consistency_report(q, temperature=0.2)
print("---")
consistency_report(q, temperature=1.2)

## 3. 可重現性：固定條件，但不要誤以為完全 deterministic

2026 新專案以 Responses API 為主。實務上要提高可重現性，先固定：模型、prompt、instructions、工具版本、temperature、資料版本與應用程式狀態；再用低溫、多次取樣、評估器或人工審核收斂。

早期 Chat Completions 有些範例會教 `seed`，但它不應再成為本課程主線控制點。對 production 來說，更重要的是把輸入、模型版本、檢索資料、工具結果和輸出一起記錄，形成可稽核軌跡。

In [ ]:
stable_prompt = "隨機講一個四字成語，只回成語。"

low_temp = ask_openai(stable_prompt, temperature=0, runs=3)
high_temp = ask_openai(stable_prompt, temperature=1.2, runs=3)

print("temperature=0   :", low_temp)
print("temperature=1.2 :", high_temp)
print("→ 低溫通常更穩定，但仍應用紀錄與評估機制處理可稽核性。")

## 4. 三家 provider 的取樣控制對照

| Provider | 隨機性參數 | 可重現策略 | 一次多樣本 | 備註 |
|----------|-----------|-----------|-----------|------|
| OpenAI Responses | `temperature`, `top_p` | 固定 model / prompt / tools / data，低溫，多次取樣後評估 | 以多次呼叫取得 | 新專案主線入口 |
| Anthropic Claude | `temperature`, `top_p`, `top_k` | 固定條件 + 低溫 + 紀錄輸入輸出 | 需多次呼叫 | 無重現保證 |
| Google Gemini | `temperature`, `top_p`, `top_k` | 固定條件 + 低溫 + 紀錄輸入輸出 | `candidate_count` | 支援 top_k |

重點不是追求「完全 deterministic」，而是建立可追蹤、可重跑、可評估的工程流程。

In [ ]:
# Anthropic：無 seed，只能靠低溫 + 多次呼叫觀察
try:
    import anthropic
    claude = anthropic.Anthropic()  # 讀取 ANTHROPIC_API_KEY
    outs = []
    for _ in range(3):
        msg = claude.messages.create(
            model='claude-haiku-4-5', max_tokens=20, temperature=1.0,
            messages=[{'role': 'user', 'content': '隨機講一個顏色，只回顏色。'}],
        )
        outs.append(msg.content[0].text.strip())
    print('Claude 三次:', outs)
except Exception as e:
    print('跳過 Claude：', type(e).__name__, e)

In [ ]:
# Gemini：用 candidate_count 一次取多個樣本
try:
    from google import genai
    from google.genai import types
    gem = genai.Client()  # 讀取 GEMINI_API_KEY
    resp = gem.models.generate_content(
        model='gemini-2.5-flash',
        contents='隨機講一個水果，只回水果。',
        config=types.GenerateContentConfig(candidate_count=3, temperature=1.0, max_output_tokens=20),
    )
    print('Gemini 三個候選:', [c.content.parts[0].text.strip() for c in resp.candidates])
except Exception as e:
    print('跳過 Gemini：', type(e).__name__, e)

## 5. 量測不確定性：logprobs 與熵 (entropy)

不只看「輸出什麼」，還能看模型「有多確定」。OpenAI 的 `logprobs` 會回傳每個 token 的候選機率，我們可由此計算**熵**：熵越高代表模型在該位置越猶豫，是一種可程式化的「信心」訊號。

In [ ]:
def entropy(distribution):
    """用簡化機率分布示範熵；實務上可用 provider 當下支援的 logprobs 或評估器取得信心訊號。"""
    return -sum(p * math.log(p) for p in distribution if p > 0)

examples = {
    "高信心分布": {"台北": 0.82, "臺北": 0.12, "北京": 0.03, "其他": 0.03},
    "開放題分布": {"牛肉麵": 0.25, "滷肉飯": 0.22, "火鍋": 0.19, "其他": 0.34},
}

for label, probs in examples.items():
    h = entropy(probs.values())
    print(label)
    for token, p in probs.items():
        print(f"  {token!r:8} p={p:.2f}")
    print(f"  → 熵 ≈ {h:.3f}（越高越不確定）\n")

## 6. 工程意涵：如何把不可控收斂回可控

理解了取樣的隨機性，後續模組的所有手段都是在「對抗」它：

| 手段 | 做什麼 | 出現在 |
|------|--------|--------|
| 降溫 + 固定條件 | 直接壓低隨機性、減少漂移 | 本章 |
| 結構化輸出 | 限制輸出空間，消除格式不確定性 | 模組 03 |
| Self-Consistency | 多次取樣後投票，抵銷單次偏差 | 模組 02 |
| best-of-N + 評分 | 取樣多個再用評估器挑最好 | 模組 07 |
| logprobs / 熵 / 評估器 | 量測信心，低信心時觸發人工介入 | 本章、模組 07 |

> **核心觀念**：你無法移除隨機性，但可以**測量它、約束它、並在它失控時偵測到**。

---

## 本章小結

1. LLM 對下一個 token 做**機率取樣**，這是非確定性的根源。
2. `temperature` / `top_p` / `top_k` 調節隨機性；溫度越高越不可預測。
3. **可重現性**：不要依賴單一 seed 參數；固定模型、prompt、工具、資料與應用狀態，並保留可稽核紀錄。
4. **量測不確定性**：可用 logprobs、熵、評估器或人工審核，把「模型有多猶豫」變成流程訊號。
5. 不可控無法消除，但可被**測量、約束、偵測**——這是整個課程的主線。